# <center>  **<span style="font-size:80px;">Exploración de los Datos</span>** </center>

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import sys
import os


from pathlib import Path

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    confusion_matrix, classification_report
)


sys.path.append(os.path.abspath(os.path.join("..")))
from src.config import DatasetKeys
from src.config import Paths
Paths.init_project()

In [ ]:
from src.water2fraud.features.preprocessor import WaterPreprocessor
preprocessor = WaterPreprocessor()

In [ ]:
seed = 80

images_path = Path("./EXTERNAL")
images_path.mkdir(parents=True, exist_ok=True)

# AMAEM

In [ ]:
df_amaem = pd.read_csv(Paths.PROC_CSV_AMAEM)

# Convertimos a periodo (YYYY-MM) directamente sobre la columna original
df_amaem[DatasetKeys.FECHA] = pd.to_datetime(df_amaem[DatasetKeys.FECHA]).dt.to_period('M')

# Instituto Nacional de Estadística (INE)

In [ ]:
def process_df(df, mapeo=None, drop=None):
    df.columns = (df.columns
                  .str.strip().str.lower()
                  .str.replace(' ', '_').str.replace(':', '')
                  .str.replace('á', 'a').str.replace('é', 'e')
                  .str.replace('í', 'i').str.replace('ó', 'o')
                  .str.replace('ú', 'u'))
    if mapeo: df = df.rename(columns=mapeo)
    if drop: df = df.drop(columns=drop)
    return df

## Municipios

In [ ]:
params_mun = {"encoding": "latin1", "sep": "\t"}
df_municipios_viviendas   = pd.read_csv(Paths.INE_MUNICIPIOS_PLAZAS, **params_mun)

### Matching Barrios - Municipios

Nuestro datos *INE* catalogan los datos a través de **municipios**. En cambio, nuestros datos *AMAEM* utilizan los **barrios**. Por tanto, deberemos asignar a cada barrio su correspondiente municipio. En caso de que no se encuentre el municipio exacto utilizaremos una ponderación.

In [ ]:
df_mapping_barrios = pd.read_csv(Paths.MAPPING_BARRIOS, sep=";")

Ahora que tenemos cada barrio mapeado con sus correspondientes municipios y ponderaciones, podremos realizar la asignación del **número de viviendas** y el **porcentaje turístico** de **viviendas turísticas** por cada barrio.

In [ ]:
def clean_ine_period(df):
    df[DatasetKeys.FECHA] = pd.to_datetime(df['Periodo'].str.replace('M', '-')).dt.to_period('M')


    if 'Total' in df.columns:
        # Convertimos a string, quitamos el punto de los miles (y si hubiera coma de decimales la pasamos a punto)
        df['Total'] = (
            df['Total'].astype(str)
            .str.replace('.', '', regex=False)  # Quitamos el punto de los miles (11.921 -> 11921)
            .str.replace(',', '.', regex=False) # Por si hay decimales con coma en el INE
        )
        # Convertimos a número real
        df['Total'] = pd.to_numeric(df['Total'], errors='coerce').fillna(0)

    return df

df_municipios_viviendas = clean_ine_period(df_municipios_viviendas)
df_municipios_viviendas = df_municipios_viviendas[
    df_municipios_viviendas['Viviendas y plazas'].str.contains('Viviendas', na=False, case=False)
]

In [ ]:
df_municipios_data = df_municipios_viviendas[['Municipios', DatasetKeys.FECHA, 'Total']].copy()
df_municipios_data.rename(columns={'Total': 'Total_vt_municipio'}, inplace=True)

# Cruzamos con los pesos
df_weighted_municipios = pd.merge(
    df_mapping_barrios, 
    df_municipios_data, 
    left_on='municipio', 
    right_on='Municipios'
)

df_weighted_municipios['peso'] = pd.to_numeric(df_weighted_municipios['peso'], errors='coerce').fillna(0)
df_weighted_municipios['Total_vt_municipio'] = pd.to_numeric(df_weighted_municipios['Total_vt_municipio'], errors='coerce').fillna(0)
df_weighted_municipios['num_vt_barrio'] = df_weighted_municipios['Total_vt_municipio'] * df_weighted_municipios['peso']

# Agrupamos por barrio y fecha (ya en formato mes-año)
df_barrio_final = df_weighted_municipios.groupby(['barrio', DatasetKeys.FECHA]).agg({
    'num_vt_barrio': 'sum'
}).reset_index()

### Porcentaje Viviendas Turísticas (INE + AMAEM)

Dado que tenemos el **Total de Viviendas Turísticas (municipio)**, podemos obtener el **Total de Viviendas Turísticas (barrio)** utilizando ponderaciones del estilo: 

**41-PLAYA DE SAN JUAN**
- 10% Alicante
- 10% Campello
- 5% San Juan Pueblo


In [ ]:
# Filtramos domésticos y seleccionamos columnas usando la fecha ya formateada
df_amaem_domestico = df_amaem[df_amaem['uso'] == 'DOMESTICO'].copy()
df_amaem_clean = df_amaem_domestico[['barrio', DatasetKeys.FECHA, 'num_contratos']]

# Cruce directo por barrio y fecha
df_final = pd.merge(df_barrio_final, df_amaem_clean, on=['barrio', DatasetKeys.FECHA], how='left')

df_final['porcentaje_turistico_real'] = (df_final['num_vt_barrio'] / df_final['num_contratos']) * 100
df_final['porcentaje_turistico_real'] = df_final['porcentaje_turistico_real'].fillna(0).round(4)

### Expansión de datos con frecuencia mensual (Interpolación)

In [ ]:
# El rango ahora se genera sobre DatasetKeys.FECHA
periodo_minimo = df_final[DatasetKeys.FECHA].min()
rango_completo = pd.period_range(start=periodo_minimo, end='2024-12', freq='M')

def expandir_y_interpolar_vt(group):
    group = group.set_index(DatasetKeys.FECHA)
    group = group.reindex(rango_completo)
    if 'num_vt_barrio' in group.columns:
        group['num_vt_barrio'] = group['num_vt_barrio'].interpolate(method='linear')
    group = group.loc['2022-01':'2024-12']
    return group

df_base = df_final[['barrio', DatasetKeys.FECHA, 'num_vt_barrio']]

df_interpolado = (
    df_base.groupby('barrio', group_keys=True)
    .apply(expandir_y_interpolar_vt, include_groups=False)
    .reset_index()
    .rename(columns={'level_1': DatasetKeys.FECHA})
)

# Merge final tras interpolación
df_resampled = pd.merge(
    df_interpolado, 
    df_amaem_clean, 
    on=['barrio', DatasetKeys.FECHA], 
    how='left'
)

df_resampled['porcentaje_turistico_real'] = (df_resampled['num_vt_barrio'] / df_resampled['num_contratos']) * 100
df_resampled['porcentaje_turistico_real'] = df_resampled['porcentaje_turistico_real'].fillna(0).round(4)

In [ ]:
df_resampled

### Merge Final e Integración (AMAEM + INE + Provincia)

In [ ]:
columnas_a_añadir = df_resampled[['barrio', DatasetKeys.FECHA, 'num_vt_barrio', 'porcentaje_turistico_real']]

df_amaem_final = pd.merge(
    df_amaem,
    columnas_a_añadir,
    on=['barrio', DatasetKeys.FECHA],
    how='left'
)

In [ ]:
df_amaem_final.isnull().sum()


In [ ]:
display(df_amaem_final.head())

In [ ]:
df_maximos = df_amaem_final.groupby('barrio')['porcentaje_turistico_real'].max().reset_index()
display(df_maximos)

## Provincia

In [ ]:
params_prov = {"encoding": "utf-8", "sep": ";"}
df_provincia_vt = pd.read_csv(Paths.INE_PROVINCIA_VT, **params_prov)

### Viviendas Trurísticas

In [ ]:
mapeo_prov = {
    "fecha": "fecha_orig", # Temporal para no chocar
    "total_numero_de_alojamientos_turisticos_ocupados": "ocupaciones_vt",
    "numero_de_noches_ocupadas": "pernoctaciones_vt",
    "estancia_media_alacant/alicante": "media_vt"
}

df_provincia_vt_proc = process_df(df_provincia_vt, mapeo=mapeo_prov)

df_provincia_vt_proc[DatasetKeys.FECHA] = pd.to_datetime(df_provincia_vt_proc['fecha_orig'].str.replace('M', '-')).dt.to_period('M')
df_provincia_vt_proc = df_provincia_vt_proc.drop(columns=['fecha_orig'])

df_prov_clean = df_provincia_vt_proc[[DatasetKeys.FECHA, 'ocupaciones_vt', 'pernoctaciones_vt']].copy()


In [ ]:
# Limpieza de puntos en miles
for col in ['ocupaciones_vt', 'pernoctaciones_vt']:
    if df_prov_clean[col].dtype == 'object':
        df_prov_clean[col] = df_prov_clean[col].str.replace('.', '', regex=False).astype(float)

# Merge final con Provincia usando DatasetKeys.FECHA
df_resultado_total = pd.merge(
    df_amaem_final,
    df_prov_clean,
    on=DatasetKeys.FECHA,
    how='left'
)

In [ ]:
display(df_resultado_total.head())

## Comunidad

In [ ]:
params_com = {"encoding": "latin1", "sep": ";"}

# Cargar archivos de Comunidad
df_comunidad_tipo_aloj = pd.read_csv(Paths.INE_COMUNIDAD_TIPO_ALOJ, **params_com)


In [ ]:
mapeo_comunidad = {
    'destino_principal': 'destino',
    'concepto_turistico': 'concepto',
    'alojamiento_principal_nivel_1': 'aloj_n1',
    'alojamiento_principal_nivel_2': 'aloj_n2',
    'motivo_principal': 'motivo',
    'transporte_principal': 'transporte',
    'forma_de_organizacion_del_viaje': 'organizacion',
    'duracion_del_viaje': 'duracion',
    'comunidad_autonoma_de_residencia': 'residencia',
    'periodo': 'periodo',
    'total': 'total'
}

drop_comunidad = ['destino', 'motivo', 'transporte', 'organizacion', 'duracion', 'residencia']

In [ ]:
df_comunidad_tipo_aloj = process_df(df_comunidad_tipo_aloj, mapeo=mapeo_comunidad, drop=drop_comunidad)

In [ ]:
df_comunidad_tipo_aloj.head(10)